In [62]:
from core.lns.operators.destroy import RandomNodeDestroy, TSPRandomEdgeDestroy
from core.lns.operators.repair import GreedyNodeRepair, RandomRepair, TSPGreedyEdgeRepair
from core.problems.tsp import TSPConfig, TSPSampler
from core.problems.tsp.solve import random_permutation

In [63]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Node destroy

In [64]:
n = 5
batch_size = 2

tsp_instance = TSPSampler(TSPConfig(num_locations=n), seed=42)(batch_size)
tsp_solution = random_permutation(tsp_instance, seed=42)

print(tsp_solution.tour)

tensor([[2, 4, 0, 1, 3],
        [4, 1, 0, 2, 3]])


In [65]:
destroyed_tsp_solution = RandomNodeDestroy(2, seed=42)(tsp_solution)
print(destroyed_tsp_solution.tour)

tensor([[ 2,  4,  0, -1, -1],
        [ 4,  1,  0, -1, -1]])


In [66]:
print(destroyed_tsp_solution.unassigned_nodes())

tensor([[1, 3],
        [2, 3]])


### Repair

In [67]:
random_repair = RandomRepair(seed=42)
_repaired = random_repair(destroyed_tsp_solution)

print(_repaired.tour)
print(_repaired.objective())

tensor([[2, 4, 0, 1, 3],
        [4, 1, 0, 2, 3]])
tensor([2.4506, 1.7551])


In [68]:
greedy_repair = GreedyNodeRepair()
_repaired = greedy_repair(destroyed_tsp_solution)

print(_repaired.tour)
print(_repaired.objective())

tensor([[2, 4, 0, 1, 3],
        [4, 1, 0, 3, 2]])
tensor([2.4506, 1.4473])


## Edge destroy

In [69]:
from core.problems.tsp.graph_solution import TSPGraphSolution

tsp_graph_solution = TSPGraphSolution.create_from_sequence(
    instance=tsp_instance, sequence=tsp_solution.tour
)

print(tsp_graph_solution.prev_nodes)
print(tsp_graph_solution.next_nodes)

tensor([[4, 0, 3, 1, 2],
        [1, 4, 0, 2, 3]])
tensor([[1, 3, 4, 2, 0],
        [2, 0, 3, 4, 1]])


In [70]:
destroyed_tsp_graph_solution = TSPRandomEdgeDestroy[TSPGraphSolution](2, seed=123)(
    tsp_graph_solution
)
print(destroyed_tsp_graph_solution.prev_nodes)
print(destroyed_tsp_graph_solution.next_nodes)

tensor([[3, 1],
        [0, 4]])
tensor([[ 4, -1,  3, -1,  2],
        [-1,  4,  0,  2, -1]])
tensor([[-1, -1,  4,  2,  0],
        [ 2, -1,  3, -1,  1]])


## Repair

In [71]:
_repaired = TSPGreedyEdgeRepair[TSPGraphSolution]()(destroyed_tsp_graph_solution)
print(_repaired.prev_nodes)
print(_repaired.next_nodes)

tensor([[4, 0, 3, 1, 2],
        [1, 4, 0, 2, 3]])
tensor([[1, 3, 4, 2, 0],
        [2, 0, 3, 4, 1]])
